In [7]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # lee el archivo .env

True

In [10]:
codigos_fred = {
    "fed_funds":    "FEDFUNDS",
    "ust10":        "GS10",
    "cpi_us":       "CPIAUCSL",
    "vix":          "VIXCLS",
    "dollar_index": "DTWEXBGS",
    "cobre_fred":   "PCOPPUSDM",
}

url_fred = "https://api.stlouisfed.org/fred/series/observations"
series_fred = {}
for nombre, codigo in codigos_fred.items():
    params = {
        "series_id": codigo,
        "api_key": os.getenv("FRED_API_KEY"),
        "file_type": "json",
        "observation_start": "2010-01-01",
    }
    r = requests.get(url_fred, params=params, timeout=30)
    r.raise_for_status()
    fred = pd.DataFrame(r.json()["observations"])
    fred["date"] = pd.to_datetime(fred["date"])
    fred["value"] = pd.to_numeric(fred["value"], errors="coerce")
    series_fred[nombre] = fred.set_index("date")["value"]
    print(f"OK {nombre}: {fred['value'].count()} observaciones")

base_fred = pd.DataFrame(series_fred)   # una columna por variable, una fila por fecha
print(base_fred.head())

OK fed_funds: 201 observaciones
OK ust10: 201 observaciones
OK cpi_us: 200 observaciones
OK vix: 4256 observaciones
OK dollar_index: 4181 observaciones
OK cobre_fred: 199 observaciones
            fed_funds  ust10   cpi_us    vix  dollar_index  cobre_fred
date                                                                  
2010-01-01       0.11   3.73  217.488    NaN           NaN    7367.375
2010-01-04        NaN    NaN      NaN  20.04       92.3566         NaN
2010-01-05        NaN    NaN      NaN  19.35       92.2236         NaN
2010-01-06        NaN    NaN      NaN  19.16       92.0941         NaN
2010-01-07        NaN    NaN      NaN  19.06       92.3684         NaN


In [9]:
codigos_bcch = {
    "dolar_observado": "F073.TCO.PRE.Z.D",
    "cuenta_corriente": "F068.A.FLU.Z.0.S.N.Z.Z.Z.Z.6.0.T",
    "tpm": "F022.TPM.TIN.D001.NO.Z.D",
    "ipc": "F074.IPC.IND.Z.EP23.C.M",
    "imacec": "F032.IMC.IND.Z.Z.EP18.Z.Z.0.M",
    "cobre_refinado": "F019.PPB.PRE.40.M",
    "exportaciones_cobre_dolares": "F068.B1.FLU.A1.0.C.N.Z.Z.Z.Z.6.0.M",
    "exportaciones_cobre_toneladas": "F034.ECUM.FLU.COCHILCO.Z.0.M",
    "total_ingresos": "F051.BT02.FLU.C.D.CLP.M",
    "total_gastos": "F051.BT03.FLU.C.D.CLP.M",
}

url_bcch = "https://si3.bcentral.cl/SieteRestWS/SieteRestWS.ashx"
series_bcch = {}
for nombre, codigo in codigos_bcch.items():
    params = {
        "timeseries": codigo,
        "user": os.getenv("BCCH_USER"),
        "pass": os.getenv("BCCH_PASS"),
        "firstdate": "2010-01-01",
        "function": "GetSeries",
    }
    r = requests.get(url_bcch, params=params, timeout=30)
    r.raise_for_status()
    bcch = pd.DataFrame(r.json()["Series"]["Obs"])
    bcch["date"] = pd.to_datetime(bcch["indexDateString"], format="%d-%m-%Y")
    bcch["value"] = pd.to_numeric(bcch["value"], errors="coerce")
    series_bcch[nombre] = bcch.set_index("date")["value"]
    print(f"OK {nombre}: {bcch['value'].count()} observaciones")

base_bcch = pd.DataFrame(series_bcch)   # una columna por variable, una fila por fecha
print(base_bcch.head())

OK dolar_observado: 4184 observaciones
OK cuenta_corriente: 66 observaciones
OK tpm: 4184 observaciones
OK ipc: 201 observaciones
OK imacec: 200 observaciones
OK cobre_refinado: 201 observaciones
OK exportaciones_cobre_dolares: 201 observaciones
OK exportaciones_cobre_toneladas: 200 observaciones
OK total_ingresos: 184 observaciones
OK total_gastos: 184 observaciones
            dolar_observado  cuenta_corriente  tpm    ipc     imacec  \
date                                                                   
2010-01-01              NaN        157.957758  NaN  57.61  72.478647   
2010-01-04           507.10               NaN  0.5    NaN        NaN   
2010-01-05           505.70               NaN  0.5    NaN        NaN   
2010-01-06           504.22               NaN  0.5    NaN        NaN   
2010-01-07           497.93               NaN  0.5    NaN        NaN   

            cobre_refinado  exportaciones_cobre_dolares  \
date                                                      
2010-01

In [ ]:
def inspeccionar(base):
    resumen = pd.DataFrame({
        "obs": base.count(),                                  # datos reales por variable
        "faltantes": base.isna().sum(),                       # vacíos por variable
        "inicio": base.apply(pd.Series.first_valid_index),    # primera fecha con dato
        "fin": base.apply(pd.Series.last_valid_index),        # última fecha con dato
    })
    print(resumen)
    print(base.describe().T.round(2))

===== FRED =====
               obs  faltantes     inicio        fin
fed_funds      201       4237 2010-01-01 2026-09-01
ust10          201       4237 2010-01-01 2026-09-01
cpi_us         200       4238 2010-01-01 2026-09-01
vix           4256        182 2010-01-04 2026-10-16
dollar_index  4181        257 2010-01-04 2026-10-16
cobre_fred     199       4239 2010-01-01 2026-07-01
               count     mean      std      min      25%      50%      75%  \
fed_funds      201.0     1.52     1.83     0.05     0.10     0.37     2.41   
ust10          201.0     2.70     1.05     0.62     1.93     2.54     3.57   
cpi_us         200.0   261.36    34.70   217.20   235.33   250.51   294.92   
vix           4256.0    18.39     6.74     9.14    13.88    16.70    20.95   
dollar_index  4181.0   109.34    12.00    85.47    94.46   113.29   119.16   
cobre_fred     199.0  7668.43  1877.61  4471.79  6235.17  7428.29  8984.68   

                   max  
fed_funds         5.33  
ust10             4.99

In [13]:
print("===== FRED =====")
inspeccionar(base_fred)
print("\n===== BCCh =====")
inspeccionar(base_bcch)

===== FRED =====
               obs  faltantes     inicio        fin
fed_funds      201       4237 2010-01-01 2026-09-01
ust10          201       4237 2010-01-01 2026-09-01
cpi_us         200       4238 2010-01-01 2026-09-01
vix           4256        182 2010-01-04 2026-10-16
dollar_index  4181        257 2010-01-04 2026-10-16
cobre_fred     199       4239 2010-01-01 2026-07-01
               count     mean      std      min      25%      50%      75%  \
fed_funds      201.0     1.52     1.83     0.05     0.10     0.37     2.41   
ust10          201.0     2.70     1.05     0.62     1.93     2.54     3.57   
cpi_us         200.0   261.36    34.70   217.20   235.33   250.51   294.92   
vix           4256.0    18.39     6.74     9.14    13.88    16.70    20.95   
dollar_index  4181.0   109.34    12.00    85.47    94.46   113.29   119.16   
cobre_fred     199.0  7668.43  1877.61  4471.79  6235.17  7428.29  8984.68   

                   max  
fed_funds         5.33  
ust10             4.99

In [12]:
print("\n===== BCCh =====")
inspeccionar(base_bcch)


===== BCCh =====
                                obs  faltantes     inicio        fin
dolar_observado                4184       1951 2010-01-04 2026-10-20
cuenta_corriente                 66       6069 2010-01-01 2026-04-01
tpm                            4184       1951 2010-01-04 2026-10-19
ipc                             201       5934 2010-01-01 2026-09-01
imacec                          200       5935 2010-01-01 2026-08-01
cobre_refinado                  201       5934 2010-01-01 2026-09-01
exportaciones_cobre_dolares     201       5934 2010-01-01 2026-09-01
exportaciones_cobre_toneladas   200       5935 2010-01-01 2026-08-01
total_ingresos                  184       5951 2010-03-01 2026-08-01
total_gastos                    184       5951 2010-03-01 2026-08-01
                                count        mean         std         min  \
dolar_observado                4184.0      699.74      160.45      455.91   
cuenta_corriente                 66.0    -2512.08     2353.65    -874

In [3]:
print(base_f.describe())
print(base_f.isna().sum())  # cuántos valores faltan por columna

        fed_funds       ust10      cpi_us          VIX  dollar_index  \
count  201.000000  201.000000  200.000000  4256.000000   4177.000000   
mean     1.515672    2.695274  261.359930    18.394182    109.332538   
std      1.832159    1.051403   34.700675     6.736809     11.999328   
min      0.050000    0.620000  217.199000     9.140000     85.469200   
25%      0.100000    1.930000  235.328500    13.880000     94.449600   
50%      0.370000    2.540000  250.509500    16.700000    113.282000   
75%      2.410000    3.570000  294.924000    20.950000    119.142000   
max      5.330000    4.990000  336.298000    82.690000    130.041300   

         cobre_fred  
count    199.000000  
mean    7668.433753  
std     1877.610200  
min     4471.787500  
25%     6235.168478  
50%     7428.289474  
75%     8984.682777  
max    13552.040909  
fed_funds       4237
ust10           4237
cpi_us          4238
VIX              182
dollar_index     261
cobre_fred      4239
dtype: int64
